# 3장 2강: 프롬프트 설계와 활용 — 실습문제


## 실습 목표

- 프롬프트의 작업 목적·입력 데이터·판단 기준·출력 형식을 구분할 수 있다.
- 모호한 분류 요청을 구체적인 Zero-shot 프롬프트로 개선할 수 있다.
- 올바른 입력·출력 예시를 골라 Few-shot 프롬프트를 구성할 수 있다.
- 정보 추출과 요약의 목적에 맞는 출력 조건을 작성할 수 있다.
- 원문과 출력 조건을 기준으로 결과를 검토하고 필요한 조건을 보완할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 아래 세 파일을 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 앞 강의를 실행하지 않아도 됩니다. 이 노트북의 준비 코드부터 순서대로 실행하세요.

| 파일 | 이번 실습에서의 활용 |
|---|---|
| 엘지유플러스_분류.csv | 분류 대상 상담, 범주, Few-shot에 사용할 입력·출력 예시 확인 |
| 엘지유플러스_요약.csv | 요약 작업 지시, 포함할 핵심 내용, 분량 조건 확인 |
| 엘지유플러스_질의응답.csv | 상담에서 확인해야 할 사실과 질문·답변 확인 |

| 컬럼 | 의미와 사용 방법 |
|---|---|
| source_id | 상담 원문 ID. 같은 상담에서 여러 작업을 만들면 반복됨 |
| consulting_category | 데이터에 부여된 상담 분야 정보 |
| task_category | 상담 요건, 길이 제한 요약, 의문사형 등 세부 작업 유형 |
| instruction | CSV에 저장된 작업 지시 또는 질문 |
| input | 분석할 상담 원문 |
| output | CSV에 저장된 출력. 원문과 비교할 참고 자료 |

### 이번 실습의 범위

교안의 명언·리뷰 분석 사례를 실제 **상담 텍스트**에 적용합니다. 실습은 **프롬프트 설계와 검토**이며, 실제 LLM API 호출과 여러 문서의 자동 처리는 포함하지 않습니다.

- 실제 CSV 값, 사람이 작성한 기대 답안, 학습용 가상 출력은 구분하여 표시합니다.
- Python 코드는 원문 조회, 프롬프트 문자열 구성, 간단한 결과 표 작성에 사용합니다. 문자열을 출력하는 것은 모델 호출이 아닙니다.
- Zero-shot과 Few-shot의 차이는 **정답이 있는 입력·출력 예시를 함께 넣는지**입니다. 범주 정의나 출력 형식을 자세히 설명해도 정답 예시가 없으면 Zero-shot입니다.
- Few-shot용 예시는 원문과 저장된 출력을 확인한 뒤 사용합니다. 현재 분류할 대상의 정답은 프롬프트에 미리 넣지 않습니다.
- 저장된 `output`을 항상 옳다고 가정하지 않습니다. 분류 판단과 요약·추출 결과는 원문과 작업 기준으로 검토합니다.
- `<NAME>`, `<DATE>`, `<CHARGE>` 같은 원문의 가림 표시는 그대로 유지합니다. 가려진 사실을 추측하여 채우지 않습니다.
- 상담 속 상품 안내는 해당 상담을 해석하기 위한 자료이며, 현재 상품 정책을 확인한 자료가 아닙니다.

**구성:** 필수 1, 필수 2, 과제 1. 과제도 필수 문제와 같은 기초~중급 수준의 프롬프트 작성과 원문 검토를 수행합니다.


## 시작 전에 읽는 개념 정리

### 1. 프롬프트는 ‘무엇을, 어떤 기준으로, 어떤 형태로’ 요청할지 정하는 것입니다

“상담을 분석해 주세요”만으로는 분류가 필요한지, 요약이 필요한지, 특정 사실을 찾으라는 것인지 분명하지 않습니다. 프롬프트를 설계한다는 것은 **원하는 분석 작업과 조건을 문장으로 명확히 전달하는 일**입니다.

이 노트북은 프롬프트를 파이썬 문자열로 만들고 내용을 확인합니다. `print()`로 화면에 표시했다고 모델이 그 지시를 실행한 것은 아닙니다. 실제 LLM/API 호출은 포함하지 않으며, 기대 답안과 가상 출력도 구분해서 읽습니다.

| 구성 요소 | 쉬운 질문 | 이번 실습에서의 예 |
|---|---|---|
| 작업 목적 | 무엇을 하게 할까요? | 상담 요건 분류, 특정 사실 추출, 상담 요약 |
| 입력 데이터 | 무엇을 분석할까요? | 해당 상담의 `input` 전체 |
| 판단 기준 | 어떤 기준으로 내용을 해석할까요? | 중심 주제가 하나인지, 구분되는 주제가 둘 이상인지 |
| 출력 형식 | 결과를 어떻게 받겠습니까? | 범주명 하나, 항목명과 값, 두 문장 요약 |

판단 기준은 **답을 고르는 기준**, 출력 형식은 **고른 답을 적는 모양**입니다. 올바른 범주를 골라도 설명 문장을 길게 덧붙이면 ‘범주명 하나만’이라는 형식 조건에 맞지 않을 수 있습니다. 반대로 허용한 범주명만 적었어도 내용에 맞지 않는 범주일 수 있습니다.

### 2. 용어를 먼저 익히기

| 용어 | 쉬운 설명 | 혼동하지 않을 점 |
|---|---|---|
| 프롬프트 | 모델에 전달할 지시사항과 입력 정보 | 이번 코드에서는 전달할 문자열을 구성하는 단계까지 수행 |
| 작업 정의 | 수행할 분석을 구체적으로 정함 | ‘분석’이라는 넓은 말보다 분류·추출·요약처럼 명확히 지정 |
| 범주 | 분류 결과로 허용하는 항목 | 이 실습의 요건 분류는 단일·다수 두 범주 |
| 선택 규칙 | 가능한 항목 중 몇 개를 어떻게 고를지 정함 | 주제가 여러 개여도 출력은 ‘다수 요건 민원’이라는 범주 하나 |
| 출력 조건 | 포함할 내용·형태·분량에 관한 요구 | 길이 조건만으로 사실의 정확성을 보장하지 않음 |
| Zero-shot | 정답이 붙은 입력·출력 예시 없이 요청하는 방식 | 조건이 자세하거나 길어도 정답 예시가 없으면 해당 |
| Few-shot | 소수의 입력·정답 출력 쌍을 함께 제시하는 방식 | 입력 원문만 여러 개 붙이는 것과는 다름 |
| 기대 답안 | 원문과 기준을 보고 사람이 정한 비교 기준 | 이번 모델이 생성한 값이 아님 |
| 가상 출력 | 비교 연습을 위해 미리 정한 응답 예 | 실제 프롬프트 실행 결과가 아님 |
| 확인 불가 | 제공된 원문으로 값을 확정할 수 없다는 표시 | 값이 0이거나 해당 사실이 없다고 확정하는 뜻은 아님 |

### 3. 세 가지 분석 작업은 필요한 조건이 다릅니다

| 작업 | 결과로 받고 싶은 것 | 지시에 적어야 할 핵심 | 결과를 확인하는 방법 |
|---|---|---|---|
| 정보 추출 | 정해진 항목의 값 | 항목 이름, 원문 근거, 정보가 없을 때 처리 | 항목별 값·근거·누락 여부 확인 |
| 분류 | 정해진 범주 | 범주 정의, 선택 규칙, 범주명 출력 조건 | 내용에 맞는 범주인지와 형식 확인 |
| 요약 | 핵심 내용을 줄인 문장 | 포함할 정보, 분량, 사실 보존 조건 | 핵심 내용·사실·문장 수·길이 확인 |

동일한 상담을 활용하더라도 정보를 네 항목으로 추출하는 것과 두 문장으로 요약하는 것은 서로 다른 작업입니다. 따라서 모든 작업에 똑같은 출력 지시를 붙이기보다 목적에 맞는 조건을 씁니다.

### 4. 이번 강의에서 사용하는 상담의 역할

| 상담 ID | 이번 실습에서의 역할 | 사용 구간 |
|---:|---|---|
| 40005 | 단일·다수 요건으로 분류할 대상 | 필수 1과 필수 2에서 동일하게 사용 |
| 40001 | 판단 기준을 보여 줄 다른 상담 예시 | 필수 2의 Few-shot 예시 후보 |
| 40006 | 판단 기준을 보여 줄 다른 상담 예시 | 필수 2의 Few-shot 예시 후보 |
| 40003 | 필요한 사실과 핵심 요약을 설계할 대상 | 과제 1 |

예시의 정답은 Few-shot 프롬프트에 넣지만, **현재 분류할 대상의 정답은 미리 넣지 않습니다**. 대상의 기대 답안은 프롬프트와 별도 변수에 보관해 결과를 검토하는 기준으로 사용합니다.

프롬프트의 개선은 글자를 늘리는 일이 아닙니다. 먼저 어떤 문제가 있는지 확인하고, 그 문제와 관련된 조건을 보완합니다. 조건을 보완한 후에도 실제 응답이 좋아졌는지는 별도로 확인해야 합니다.


## 실습 준비

아래 **제공 코드**를 그대로 실행하세요. 파일 조회 함수 작성 자체는 평가하지 않습니다.

1. 세 파일의 크기·컬럼·자료형·결측치·빈 문자열을 확인합니다.
2. 고유 상담 수와 작업 유형 분포, 본문 길이를 확인합니다.
3. 상담 ID 40001·40003·40005·40006을 이번 실습 대상으로 확인합니다.
4. `get_record()`로 상담 ID와 작업 유형, 필요한 경우 질문까지 지정하여 한 행을 조회합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 실습용 식별자입니다. 원본 파일의 컬럼이 아니며 파일 순서가 바뀌면 달라집니다. 같은 상담 ID에 여러 작업이 있으므로 `source_id`만으로 특정 작업 행을 고르지 않습니다.

본문 길이는 데이터 규모를 살펴보기 위한 **글자 수**입니다. LLM의 토큰 수나 API 사용량을 계산한 값은 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40001, 40003, 40005, 40006]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])


### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


---
## 필수 1. 프롬프트 기본 구성과 Zero-shot 설계

### 문제 1-1. 모호한 분석 요청을 분류 프롬프트로 바꾸기

#### 문제 설명

상담 ID **40005**의 `상담 요건`을 분류하려고 합니다. 다음 요청은 구체적인 작업과 출력 조건이 부족합니다.

```text
다음 상담을 분석해 주세요.

상담 원문:
[40005의 input 전체]
```

사용할 범주는 CSV의 지시에 있는 `단일 요건 민원`, `다수 요건 민원`입니다. 다음은 이번 실습에 적용할 판단 기준입니다. 데이터 전체의 공식 라벨 정의를 새로 확정하는 것은 아닙니다.

- **단일 요건 민원:** 중심 요청·주제가 하나인 상담
- **다수 요건 민원:** 서로 구분되는 요청·주제가 두 가지 이상인 상담
- 같은 요청의 절차·준비물·처리 시점에 관한 세부 질문은 하나의 요건으로 봅니다.
- 다른 서비스 문제와 별도의 할인 상담이 함께 논의되면 각각의 주제를 확인합니다.
- 문장 수나 질문 횟수만으로 판단하지 않습니다.

#### 요구사항

1. `get_record(classification, 40005, "상담 요건")`으로 `target_row`를 선택하고 원문·기존 지시·저장된 출력을 확인하세요.
2. 위 모호한 요청에 원문 전체를 붙인 `weak_prompt`를 작성하세요. `weak_prompt = "다음 상담을 분석해 주세요.\n\n상담 원문:\n" + target_row["input"]`을 사용해도 됩니다.
3. 작업 목적, 입력 데이터, 판단 기준, 출력 형식을 모두 포함한 `zero_shot_prompt`를 작성하세요. 두 범주 중 **이름 하나만** 출력하도록 지정합니다. 정답 예시나 대상 상담의 저장된 `output`은 넣지 않습니다.
4. 네 구성 요소를 `구성 요소`, `프롬프트에 작성한 내용` 컬럼의 `prompt_components` 표로 정리하세요.
5. 원문을 읽고 기대 분류와 근거를 **프롬프트 밖에** 별도로 적으세요.

**보조 구문:** 여러 줄 문자열은 `"""..."""`로 작성할 수 있습니다. 문자열끼리 `+`로 연결하고 `print(프롬프트)`로 전체 내용을 확인합니다. 모델을 호출하는 코드는 작성하지 않습니다.

#### 해석 질문

**Q1.** 모호한 요청에는 무엇이 부족하며, 새 프롬프트의 네 구성 요소는 각각 무엇인가요?

**Q2.** 원문에 근거한 기대 분류는 무엇인가요? 질문 문장 수만 세거나, 실제로 완료된 업무만 세면 어떤 문제가 있나요?

**Q3.** 자세한 범주 정의와 출력 조건을 넣었는데도 Zero-shot인 이유는 무엇인가요? 대상 상담의 저장된 정답을 프롬프트에 넣지 않는 이유는 무엇인가요?

**Q4.** `다수 요건 민원`, `다수 요건으로 보입니다.`, `분류: 다수 요건 민원`이 나온다면 어떤 출력이 요구 형식에 맞나요? 프롬프트를 작성했다는 것만으로 실제 품질 개선을 확인한 것인가요?

#### 제출 결과

- 원문 조회 코드, `weak_prompt`, `zero_shot_prompt`
- `prompt_components` 표
- 기대 분류와 Q1~Q4 답변


### 풀이 전 안내 — 네 질문에 답하는 지시 작성하기

‘무엇을 할 것인가’, ‘무엇을 분석할 것인가’, ‘어떤 기준을 적용할 것인가’, ‘어떤 형태로 받을 것인가’를 먼저 짧게 적어 보세요. 이를 하나의 프롬프트로 연결하고 원문 전체가 실제 문자열에 포함되어 있는지 출력하여 확인합니다.

범주 이름만 적지 말고 두 범주의 차이와 세부 질문을 처리할 기준도 전달하세요. ‘다수’ 범주를 선택하는 것과 여러 범주명을 동시에 출력하는 것은 다릅니다. 문제의 출력 조건은 이름 하나입니다.

기대 분류와 근거는 프롬프트 밖에 작성하세요. 범주 정의와 출력 형식을 알려 주는 것은 가능하지만, 대상의 정답을 미리 제공하지 않는다는 조건을 지켜야 합니다.


In [ ]:
# 필수 1 코드를 작성하세요.


### 필수 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


---
## 필수 2. Few-shot 예시 선택과 비교

### 문제 2-1. 잘못된 예시를 고치고 Few-shot 구성하기

#### 문제 설명

필수 1과 같은 상담 **40005**를 같은 기준으로 분류합니다. 정답 예시 후보는 다음과 같습니다. 두 번째 후보에는 검토를 위한 오류를 넣었습니다.

| 예시 상담 ID | 가져올 본문 | 후보에 적힌 출력 |
|---:|---|---|
| 40001 | 분류 파일의 상담 요건 행 input | 다수 요건 민원 |
| 40006 | 분류 파일의 상담 요건 행 input | 다수 요건 민원 |

원문과 저장된 분류를 확인한 뒤 예시를 수정하세요. 40001은 여러 서비스 문의가 함께 있는 상담이고, 40006은 휴대폰 해지 방법·서류·처리 시점을 묻는 상담입니다.

또한 다음은 **같은 평가 대상 40005**에 대한 가상 비교 자료입니다. 실제 모델 실험 결과가 아니며, 실제로 이런 결과가 나왔다고 가정했을 때 어떻게 읽을지 연습합니다.

| 방식 | 가상 출력 |
|---|---|
| Zero-shot | 단일 요건 민원 |
| Few-shot | 다수 요건 민원 |

#### 요구사항

1. 40001·40006의 `상담 요건` 행을 각각 `example_multi`, `example_single`로 가져와 원문과 저장된 출력을 확인하세요.
2. `source_id`, `후보 출력`, `검토 후 출력`, `원문 근거`를 담은 `example_review` 표를 작성하세요. 잘못된 후보를 수정합니다.
3. 필수 1의 `classification_rules`를 유지하고, 검토한 두 상담의 **본문 전체와 올바른 분류**를 입력·출력 쌍으로 추가하여 `few_shot_prompt`를 작성하세요. 마지막에 40005의 본문과 빈 `분류:` 표시를 넣습니다. 대상의 정답은 넣지 않습니다.
4. 가상 비교 자료에 `허용 범주 여부`, `기대 분류와 일치`를 추가한 `comparison_table`을 작성하세요. 허용 범주는 두 개이며, 기대 분류는 필수 1에서 원문으로 판단한 값을 사용합니다.
5. 단순히 문자열이 일치하는지 확인한 결과와 실제 모델의 성능을 구분하여 해석하세요.

**보조 구문:** `문자열 + "\n" + 다른문자열`, `.isin([허용 범주])`, `표["가상 출력"] == expected_class`

#### 해석 질문

**Q1.** 어떤 예시 후보를 어떻게 수정해야 하나요? 해지 방법과 처리 시점을 따로 질문했다고 각각 별개의 요건으로 세면 왜 문제가 되나요?

**Q2.** 두 프롬프트의 공통 조건과 차이는 무엇인가요? 추가한 예시는 어떤 판단 기준을 보여 주나요?

**Q3.** 가상 비교 표에서 형식과 내용이 맞는 경우를 각각 설명하세요. 이 표를 실제 Few-shot 성능 향상의 근거라고 말할 수 있나요?

**Q4.** Zero-shot으로 충분한 결과를 얻는 경우와, 단일·다수 요건을 반복해서 혼동하는 경우에 각각 어떻게 접근하나요? Few-shot 예시를 고를 때 확인할 점을 세 가지 쓰세요.

#### 제출 결과

- 두 예시 상담의 원문 조회와 `example_review`
- `few_shot_prompt` 전체
- 가상 `comparison_table`과 Q1~Q4 답변


### 풀이 전 안내 — 예시는 붙이기 전에 검토하기

각 예시의 상담 요건 행을 조회하고 전체 원문을 읽으세요. 후보에 적힌 출력과 저장된 출력을 참고하되, 이번 기준으로 왜 맞고 틀린지 근거를 설명해야 합니다. 질문 개수만 세지 말고 같은 요청의 세부 절차인지 별개의 주제인지 나누어 보세요.

Few-shot에는 필수 1의 기준을 유지한 뒤, 검토한 예시의 본문과 정답을 짝지어 넣습니다. 마지막에는 현재 분류 대상의 본문과 답을 작성할 위치만 둡니다. 예시 정답과 대상 정답을 혼동하지 마세요.

가상 비교표의 `.isin()`은 허용 범주명 검사를, `== expected_class`는 준비한 기대 답안과의 문자열 비교를 수행합니다. 이 결과를 실제 모델 실행 결과로 설명하지 않습니다.


In [ ]:
# 필수 2 코드를 작성하세요.


### 필수 2 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


---
## 과제 1. 정보 추출·요약 목적에 맞는 프롬프트 설계

### 문제 3-1. 같은 상담에서 필요한 사실과 요약 조건 정하기

#### 문제 설명

상담 ID **40003**을 대상으로, 필수 문제에서 익힌 네 구성 요소를 정보 추출과 요약에 적용합니다. 다음 행을 확인하세요.

| 변수명 | 파일 | task_category | instruction 선택 조건 |
|---|---|---|---|
| owner_row | qa | 의문사형 | 고객의 결합 할인은 누구 앞으로 되어 있나요? |
| keep_row | qa | 예/아니요형 | 고객의 요금제 변경 시 기존 결합 할인은 유지되나요? |
| short_summary_row | summarization | 길이 제한 요약 | 추가 조건 없음 |

정보 추출 항목은 **변경한 요금제, 기존 결합 할인 유지 여부, 결합 할인 지정 대상, 변경 후 정확한 월 요금(원)**입니다. 원문에서 확인되지 않는 값은 `확인 불가`로 적도록 설계합니다.

요약은 **2문장, 300자 이내**로 작성하도록 설계하고, **요금제 변경 처리, 기존 결합 할인 유지 여부, 멤버십 등급 변경**을 반드시 포함하도록 지정합니다. 원문에 없는 사실을 추가하거나 의미를 바꾸지 않아야 합니다.

다음은 프롬프트의 보완점을 찾기 위한 **가상 오류 결과**입니다. 실제 모델을 실행한 결과가 아닙니다.

```text
[정보 추출의 일부]
결합 할인 지정 대상: 고객 본인
변경 후 정확한 월 요금(원): 37500

[요약]
고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다.
```

#### 요구사항

1. 지정한 세 행을 조회하고, 상담 원문은 한 번, 지시와 저장된 출력은 각각 출력하세요. 세 `input`이 같은지도 확인합니다.
2. 네 추출 항목, 원문 근거 조건, `확인 불가` 처리 기준, `항목명: 값` 출력 형식을 갖춘 `extraction_prompt`를 작성하세요. 원문 전체를 붙입니다.
3. 포함할 세 가지 핵심 정보, 분량, 사실 보존 조건을 갖춘 `summary_prompt`를 작성하세요. 원문 전체를 붙입니다. 두 프롬프트 모두 정답 예시를 넣지 않습니다.
4. 원문을 읽고 정보 추출의 기대 답안과 2문장 요약을 **프롬프트 밖에** 직접 작성하세요. 요약 길이는 `len(요약문)`으로 확인할 수 있습니다.
5. 가상 오류 결과를 `문제 부분`, `원문에 따른 수정`, `보완할 프롬프트 조건` 컬럼의 `improvement_table`로 검토하세요. 정보 오류뿐 아니라 요약의 핵심 정보 누락과 분량 조건도 확인합니다.

#### 해석 질문

**Q1.** 정보 추출과 요약의 목적·출력 형태·주요 조건은 어떻게 다른가요? 이번 과제의 두 프롬프트는 Zero-shot인가요, Few-shot인가요?

**Q2.** 추출 항목 네 개의 기대 값과 각각의 근거를 설명하세요. 원문에 `5G 다이렉트 37.5`라는 표현이 있다고 변경한 요금제의 정확한 월 요금을 37,500원으로 채워도 되나요?

**Q3.** 가상 오류 결과에서 무엇을 수정하고 프롬프트의 어떤 조건을 보완해야 하나요? 2문장·300자 이내라는 형식만 맞으면 좋은 요약인가요?

**Q4.** 프롬프트 수정 전후를 어떤 데이터와 기준으로 비교해야 하나요? 원문에 금액이 없으면 예시를 추가하거나 지시를 길게 쓰는 것만으로 정확한 금액을 만들 수 있나요?

#### 제출 결과

- 세 행 조회와 원문 동일성 확인
- `extraction_prompt`, `summary_prompt`
- 사람이 작성한 추출 기대 답안과 2문장 요약
- `improvement_table`과 Q1~Q4 답변


### 풀이 전 안내 — 작업마다 조건을 다르게 구성하기

정보 추출에는 네 항목과 각 항목의 근거 조건, 확인할 수 없을 때의 처리 방법을 적으세요. 요약에는 반드시 담을 세 핵심 내용과 두 문장·300자 이내 조건을 적습니다. 두 프롬프트에 같은 원문을 사용하더라도 결과 형태는 다르게 요청합니다.

원문에 여러 요금제가 나올 때는 문의만 한 것과 실제로 변경한 것을 구분하세요. 가려진 금액이나 날짜는 주변의 다른 숫자로 채우지 않습니다. 기대 답안은 원문을 읽어 프롬프트 밖에 작성합니다.

오류 검토표에서는 ‘어떤 부분이 잘못되었는지’, ‘원문 기준으로 어떻게 고칠지’, ‘그 문제를 줄이려면 어떤 지시를 명확히 할지’를 한 행에 연결하세요. 요약은 내용과 형식 모두 확인하며, `len()`은 글자 수만 계산한다는 점에 유의합니다.


In [ ]:
# 과제 1 코드를 작성하세요.


### 과제 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


## 개념 정리 — 이번 강의에서 구분할 것

| 구분 | 첫 번째 의미 | 두 번째 의미 |
|---|---|---|
| 작업 목적 / 입력 데이터 | 무엇을 수행할지 | 무엇을 분석할지 |
| 판단 기준 / 출력 형식 | 어떤 의미 기준으로 답할지 | 결과를 어떤 모양으로 적을지 |
| 범주 정의 / 정답 예시 | 범주가 뜻하는 내용 | 특정 입력을 어떤 범주로 판단했는지 보여 주는 사례 |
| Zero-shot / Few-shot | 정답 예시 없이 요청 | 입력·정답 출력 쌍을 함께 제시 |
| 예시 상담 / 분류 대상 | 판단 사례를 보여 주는 자료 | 지금 답을 얻고 싶은 자료 |
| 프롬프트 문자열 / 모델 응답 | 모델에 전달할 요청 내용 | 실제 호출 뒤 생성된 결과 |
| 기대 답안 / 가상 출력 | 원문을 보고 사람이 정한 비교 기준 | 비교 연습을 위해 미리 정한 결과 |
| 허용 범주 여부 / 기대 분류 일치 | 허용한 범주명인지 검사 | 준비한 기대 범주와 같은지 검사 |
| 확인 불가 / 값이 0 | 제공된 자료로 확정할 수 없음 | 숫자가 0이라는 구체적인 값 |
| 조건 보완 / 품질 개선 확인 | 필요한 지시를 더 명확히 작성 | 실제 응답을 같은 기준으로 다시 비교 |

설명은 ‘어떤 작업을 요청했는가 → 어떤 기준과 형식을 정했는가 → 원문에 어떤 근거가 있는가 → 결과를 무엇으로 확인할 것인가’ 순서로 작성해 보세요. 프롬프트를 길게 썼다는 사실보다 각 문구가 어떤 모호함이나 오류를 줄이려는지 설명하는 것이 중요합니다.


---
## 실습 마무리

다음 질문에 자신의 말로 답하세요.

1. 이번 실습에서 프롬프트로 해결하려던 분석 문제는 무엇인가요?
2. 네 구성 요소를 어떻게 명확히 했나요?
3. Zero-shot과 Few-shot을 구분하는 기준과 선택 기준은 무엇인가요?
4. 정보 추출·분류·요약 결과는 각각 무엇을 근거로 검증하나요?
5. 이번에 실제 실행한 작업과 아직 확인하지 않은 결과는 무엇인가요?


### 실습 마무리 답변 작성란

1.

2.

3.

4.

5.
